# Geographic Prediction with Random Forests

## Load libraries

In [ ]:
import numpy as np
import allel
from tqdm import tqdm
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json
from typing import List

## Loading the data

In [ ]:
metadata = pd.read_csv("metadata_cleaned.csv")
metadata

## Visualizing Data Distributions

In [ ]:
px.bar(
    metadata['country'].value_counts(),
    template='simple_white',
    title='Number of samples per country'
)

In [ ]:
px.bar(
    metadata['Region'].value_counts(),
    template='simple_white',
    title='Number of samples per region'
)

## Split data into training and test sets

In [ ]:
# split data into 80:20 train test
from sklearn.model_selection import train_test_split
train_metadata, test_metadata = train_test_split(metadata, test_size=0.2, random_state=42, stratify=metadata['Region'])

In [ ]:
input_file = 'variants.vcf.gz'
callset = allel.read_vcf(input_file)

In [ ]:
def subset_callset_by_sample(callset: dict,samples: List[str]) -> dict:
    sample_index = [i for i,s in enumerate(callset['samples']) if s in samples]
    return {
        "samples":callset['samples'][sample_index],
        "calldata/GT": callset['calldata/GT'][:,sample_index,:],
        "variants/ALT": callset['variants/ALT'],
        "variants/CHROM": callset['variants/CHROM'],
        "variants/FILTER_PASS": callset['variants/FILTER_PASS'],
        "variants/ID": callset['variants/ID'],
        "variants/POS": callset['variants/POS'],
        "variants/QUAL": callset['variants/QUAL'],
        "variants/REF": callset['variants/REF'],
        "variants/GENE": callset.get('variants/GENE',None),
        "variants/AA": callset.get('variants/AA',None),
    }

train_callset = subset_callset_by_sample(callset,train_metadata['sample'].tolist())
test_callset = subset_callset_by_sample(callset,test_metadata['sample'].tolist())

In [ ]:
rows = []
for i in tqdm(range(len(train_callset['variants/POS']))):
    row = [x[1] if x[1]!=-1 else np.nan for x in train_callset['calldata/GT'][i]]
    rows.append(row)

X_train = np.array(rows).T

In [ ]:
sample2region = dict(zip(metadata['sample'],metadata['Region']))
y_train = [sample2region[s] for s in train_callset['samples']]



## Set up Random Forest model

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
clf = HistGradientBoostingClassifier(max_iter=100).fit(X_train, y_train)
clf.score(X_train, y_train)

In [ ]:
rows = []
for i in tqdm(range(len(test_callset['variants/POS']))):
    row = [x[1] if x[1]!=-1 else np.nan for x in test_callset['calldata/GT'][i]]
    rows.append(row)

X_test = np.array(rows).T

y_test = [sample2region[s] for s in test_callset['samples']]

clf.score(X_test, y_test)

## Look at prediction by country

In [ ]:
sample2country = dict(zip(metadata['sample'],metadata['country']))

In [ ]:
y_country_train = np.array([sample2country[s] for s in train_callset['samples']])
clf_country = HistGradientBoostingClassifier(max_iter=100).fit(X_train, y_country_train)

In [ ]:
y_country_test = np.array([sample2country[s] for s in test_callset['samples']])
clf_country.score(X_test, y_country_test)

## Probability visualisation

In [ ]:
geojson = json.load(open("world-fixed.geojson"))

In [ ]:
df = pd.DataFrame({'country':clf_country.classes_,'probability':clf_country.predict_proba(X_test)[203]})
subgeojson = {'type': 'FeatureCollection', 'features': [x for x in geojson['features'] if x['properties']['ADMIN'] in df['country'].values]}

In [ ]:
fig = go.Figure(data=go.Choropleth(
    locations=df['country'],locationmode="country names",
    z=df['probability'], colorscale='Reds',
    marker_line_color='darkgray',
    marker_line_width=0.5,
    )
)
fig.update_layout(
    title_text='Geographic source probability',
    geo=dict(
        showframe=False,
        showcoastlines=False,
        projection_type='equirectangular'
    ),
)
fig.update_layout(
    margin=dict(l=20, r=20, t=60, b=20),
)
fig.show()

In [ ]:
np.where(pd.Series(y_country_test)=='Thailand')

## Using the model

In [ ]:
def get_input_features():
    return list(zip(callset['variants/CHROM'], callset['variants/POS'], callset['variants/ALT'][:,0]))

input_features = get_input_features()
input_features

In [ ]:
variants = {}
import pysam
vcf = pysam.VariantFile('test.vcf')
for var in vcf:
    key = (var.chrom, var.pos,var.alts[0])
    if key in input_features:
        variants[key] = var.samples[0]['GT'][0]

input_data = list(variants.values())
input_data


In [ ]:
probs = clf.predict_proba([input_data])
tab = pd.DataFrame(zip(clf.classes_,probs[0]))
tab.columns = ['Region','Probability']
tab


In [ ]:
class GeoPredictor:
    def __init__(self, model: HistGradientBoostingClassifier, input_features: List[tuple]):
        self.model = model
        self.input_features = input_features

    def predict(self, vcf_file: dict) -> dict:
        variants = self.get_variants_from_vcf(vcf_file)
        input_data = [variants.get(feat, np.nan) for feat in self.input_features]
        probs = self.model.predict_proba([input_data])[0]
        tab = pd.DataFrame(zip(self.model.classes_,probs))
        tab.columns = ['Region','Probability']
        return tab.sort_values(by='Probability', ascending=False)

    def get_variants_from_vcf(self, filename):
        vcf = pysam.VariantFile(filename)
        variants = {}
        for var in vcf:
            key = (var.chrom, var.pos,var.alts[0])
            if key in self.input_features:
                variants[key] = var.samples[0]['GT'][0]
        return variants

In [ ]:
import pickle
gp = GeoPredictor(clf, input_features)
pickle.dump(gp, open('geopredictor_model.pkl','wb'))